# 07 · Asynchronous Testing

**Focus:** testing `async` / `await` code with `pytest-asyncio`.

A coroutine (`async def`) doesn't run when you call it — it returns a coroutine object that must be
*awaited* inside an event loop. Plain pytest can't `await` your test, so it would just skip the body.
The **`pytest-asyncio`** plugin bridges this: it runs `async def test_*` functions inside an event loop.

Two ways to opt in:
- Mark each async test with **`@pytest.mark.asyncio`**, or
- Set **`asyncio_mode = auto`**, which treats *every* `async def test_*` as an asyncio test — no marker needed.

We'll show both. To keep this notebook self-contained (no shared config file), we pass the mode on the
command line with `-o asyncio_mode=auto`.

### Setup — point the shell at this course's tools
The `!` cells below run command-line tools (`pytest`, later `mutmut`, `playwright`). We prepend the
active kernel's `bin/` directory to `PATH` so those commands resolve to the versions installed for
**this course**, not some other Python on your machine. Run this cell first.

In [1]:
import sys, os
# The kernel's own interpreter lives in the course virtualenv's bin/ directory.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
print("Using Python:", sys.executable)
!pytest --version

Using Python: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python


pytest 9.1.1


### An async data-fetching coroutine
We simulate latency with `asyncio.sleep` so it's genuinely asynchronous — but deterministic.

In [2]:
%%writefile nb07_fetch.py
import asyncio

async def fetch_user(user_id: int) -> dict:
    '''Pretend to hit a network; returns after a tiny async delay.'''
    await asyncio.sleep(0.01)          # simulated I/O wait
    return {"id": user_id, "name": f"user_{user_id}"}

async def fetch_all(ids: list[int]) -> list[dict]:
    '''Fetch many users concurrently with asyncio.gather.'''
    return await asyncio.gather(*(fetch_user(i) for i in ids))

Writing nb07_fetch.py


### Async tests
Note the test functions themselves are `async def` and use `await`. The first uses the explicit
marker; the rest rely on `asyncio_mode=auto` (passed on the CLI below).

In [3]:
%%writefile test_nb07.py
import pytest
from nb07_fetch import fetch_user, fetch_all

@pytest.mark.asyncio
async def test_fetch_one_explicit_marker():
    result = await fetch_user(42)
    assert result == {"id": 42, "name": "user_42"}

async def test_fetch_one_auto_mode():
    # No marker needed when asyncio_mode=auto.
    result = await fetch_user(7)
    assert result["name"] == "user_7"

async def test_fetch_all_concurrent():
    results = await fetch_all([1, 2, 3])
    assert [r["id"] for r in results] == [1, 2, 3]
    assert len(results) == 3

Writing test_nb07.py


### Run it with `asyncio_mode=auto`
`-o asyncio_mode=auto` sets the config option inline, so both the marked and unmarked async tests run.

In [4]:
!pytest -v -o asyncio_mode=auto test_nb07.py

============================= test session starts ==============================


platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 3 items                                                              

test_nb07.py::test_fetch_one_explicit_marker PASSED                      [ 33%]
test_nb07.py::test_fetch_one_auto_mode PASSED                            [ 66%]
test_nb07.py::test_fetch_all_concurrent 

PASSED                           [100%]

============================== 3 passed in 0.06s ===============================


### ⚠️ Common pitfall — a forgotten `await` fails silently
Calling a coroutine without `await` doesn't run it — it just creates a coroutine object and emits a
`RuntimeWarning: coroutine '...' was never awaited`. Because the object is truthy, naive assertions can
still "pass", hiding the bug. **Treat that warning as an error in async tests.**

### 🔬 Try it yourself
The test below *forgets* the `await`. **Predict:** does it pass or fail — and what warning appears at
the bottom of the report? Run it, read the `RuntimeWarning`, then add `await` before `fetch_user(1)`
and re-run to see the warning vanish.

In [5]:
%%writefile test_nb07_tryit.py
from nb07_fetch import fetch_user

async def test_forgot_await():
    result = fetch_user(1)          # BUG: missing `await` -> this is a coroutine, not a dict
    assert result is not None       # trivially true, so the test "passes" despite the bug

Writing test_nb07_tryit.py


In [6]:
!pytest -v -o asyncio_mode=auto test_nb07_tryit.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item                                                               

test_nb07_tryit.py::test_forgot_await PASSED                             [100%]



=============================== warnings summary ===============================
test_nb07_tryit.py::test_forgot_await
  /opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/events.py:88: RuntimeWarning: coroutine 'fetch_user' was never awaited
    self._context.run(self._callback, *self._args)
  Enable tracemalloc to get traceback where the object was allocated.
  See https://docs.pytest.org/en/stable/how-to/capture-warnings.html#resource-warnings for more info.

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
========================= 1 passed, 1 warning in 0.03s =========================


### What you learned
- Async tests are `async def` functions that `await` your coroutines.
- `@pytest.mark.asyncio` opts a single test in; `asyncio_mode=auto` opts in *all* async tests.
- `-o asyncio_mode=auto` sets that option on the command line (equivalently, put it in `pytest.ini` /
  `pyproject.toml`).
- `asyncio.gather` lets you test concurrent fetches just like production code.

Next up: **integration testing & snapshots** with FastAPI + syrupy.